<a href="https://colab.research.google.com/github/NicoAndreas/Branch-html/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Klasifikasi & Klastering — Telco Customer Churn

**Mata Kuliah:** Machine Learning  
**Program Studi:** S-1 Teknik Informatika  
**Bentuk:** Praktikum kelompok (3–4 orang)

---

## 👥 Anggota Kelompok

| No | Nama | NIM |
|---:|------|-----|
| 1  |      |     |
| 2  |      |     |
| 3  |      |     |
| 4  |      |     |

## 🎯 Tujuan Pembelajaran

1. **Klastering** — segmentasi pelanggan dengan algoritma *unsupervised*.
2. **Klasifikasi** — memprediksi *churn* dengan algoritma *supervised*.
3. **Integrasi** — menggabungkan keduanya & merefleksikan kapan masing-masing tepat digunakan.

## 📋 Cara Mengerjakan

1. Klik tombol **Open in Colab** di README repo tim Anda.
2. Di Colab, pilih `File > Save a copy in GitHub` ke repo tim (branch `main`).
3. Kerjakan setiap sel berlabel `# TODO` **secara berurutan**.
4. **JANGAN HAPUS** sel markdown panduan. Boleh menambah sel baru.
5. Lengkapi laporan di `reports/laporan.md` setelah notebook selesai.

> 💡 **Tip:** Setiap TODO sudah dipecah menjadi langkah kecil. Jika satu TODO membingungkan, baca dulu sel markdown di atasnya — petunjuknya ada di situ.

---

## 0 · Setup

Import library dan set `random_state` agar hasil **reproducible**.

In [ ]:
# === 0.1 Imports & Konstanta ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import (
    silhouette_score,
    classification_report,
    roc_auc_score,
    RocCurveDisplay,
    ConfusionMatrixDisplay,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_style("whitegrid")

print("Setup selesai. Lanjut ke sel berikutnya.")

### 0.2 Memuat dataset

Dataset: **Telco Customer Churn** (7.043 baris, 21 kolom).  
Sumber asli: <https://www.kaggle.com/datasets/blastchar/telco-customer-churn>

Sel di bawah akan:
1. Mencari file lokal di working directory.
2. Jika tidak ada, mencoba memuat dari URL mirror.
3. Jika keduanya gagal, **upload manual**: panel kiri Colab → ikon **Files** → **Upload to session storage** → pilih CSV-nya.

Petunjuk lengkap ada di `data/README.md`.

In [ ]:
# === 0.2 Load dataset ===
import os

LOCAL_PATHS = [
    "Telco-Customer-Churn.csv",
    "WA_Fn-UseC_-Telco-Customer-Churn.csv",
    "data/Telco-Customer-Churn.csv",
    "data/WA_Fn-UseC_-Telco-Customer-Churn.csv",
]
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df = None
for path in LOCAL_PATHS:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"✓ Dataset dimuat dari file lokal: {path}")
        break

if df is None:
    try:
        df = pd.read_csv(URL)
        print("✓ Dataset dimuat dari URL fallback.")
    except Exception as e:
        raise FileNotFoundError(
            "Tidak bisa memuat dataset. Silakan upload manual ke Colab "
            "(panel kiri > Files > Upload) lalu jalankan ulang sel ini.\n"
            f"Detail error: {e}"
        )

print(f"Shape: {df.shape}")
df.head()

---

## 1 · EDA Singkat

Tujuan: memahami struktur data sebelum modeling.

> ⚠️ **Pitfall sering kelewat:** kolom `TotalCharges` mungkin bertipe `object` (bukan `float`)! Ada beberapa baris berisi spasi kosong yang harus di-handle saat preprocessing. Perhatikan di TODO 1.1 di bawah.

In [ ]:
# === TODO 1.1 — Cek struktur dataset ===
# Petunjuk method: df.info(), df.isnull().sum(), df.describe()

# (a) Tampilkan info dtypes & jumlah non-null per kolom
# TODO: tulis kode Anda


# (b) Hitung total missing values per kolom (sort descending)
# TODO: tulis kode Anda


# (c) Tampilkan ringkasan statistik fitur numerik
# TODO: tulis kode Anda


In [ ]:
# === TODO 1.2 — Distribusi target (Churn) ===
# Karena target tidak seimbang, kita perlu tahu proporsi awalnya.
# Petunjuk: df["Churn"].value_counts(normalize=True)

# (a) Hitung proporsi target
# TODO: tulis kode Anda


# (b) Plot bar chart distribusi target
# Petunjuk: sns.countplot(data=df, x="Churn")
# TODO: tulis kode Anda


In [ ]:
# === TODO 1.3 — Eksplorasi 2 fitur kunci vs Churn ===
# Pilih MINIMAL 2 fitur (1 numerik + 1 kategorik) lalu visualisasikan kaitannya dengan Churn.
#
# Contoh ide:
#   - Numerik: tenure / MonthlyCharges  → boxplot per Churn
#     sns.boxplot(data=df, x="Churn", y="tenure")
#   - Kategorik: Contract / InternetService → countplot dengan hue
#     sns.countplot(data=df, x="Contract", hue="Churn")

# (a) Visualisasi fitur numerik vs Churn
# TODO: tulis kode Anda


# (b) Visualisasi fitur kategorik vs Churn
# TODO: tulis kode Anda


### TODO 1.4 — Refleksi EDA

> ✏️ **Isi MINIMAL 3 poin observasi** dari EDA di atas. Contoh format:
>
> 1. Distribusi target tidak seimbang (~26.5% churn vs ~73.5% tidak churn) → akan berpengaruh ke pemilihan metrik evaluasi.
> 2. _____
> 3. _____

---

## 2 · Preprocessing

Bersihkan data agar siap masuk ke algoritma ML.

In [ ]:
# === TODO 2.1 — Bersihkan kolom TotalCharges ===
# Kolom ini sering bertipe object karena ada baris dengan nilai " " (spasi kosong).
#
# Petunjuk:
#   - Konversi: pd.to_numeric(df["TotalCharges"], errors="coerce")
#     → NaN akan muncul di baris yang tadinya spasi.
#   - Strategi handle NaN: pilih SALAH SATU — drop row, atau imputasi median.
#     Tulis komentar singkat alasan pilihan Anda.

# (a) Konversi TotalCharges ke numeric (NaN-coerced)
# TODO: tulis kode Anda


# (b) Cek berapa NaN yang muncul
# TODO: tulis kode Anda


# (c) Handle NaN (drop ATAU imputasi median)
# TODO: tulis kode Anda + komentar singkat alasan pilihan


In [ ]:
# === TODO 2.2 — Drop kolom non-prediktif & encode target ===

# (a) Drop kolom 'customerID' (bukan fitur prediktif, hanya identifier unik)
# TODO: tulis kode Anda


# (b) Encode target Churn: 'Yes' → 1, 'No' → 0
# Petunjuk: df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
# TODO: tulis kode Anda


# (c) Verifikasi: tampilkan df["Churn"].value_counts()
# TODO: tulis kode Anda


In [ ]:
# === TODO 2.3 — One-hot encoding fitur kategorik ===
# Pisahkan X (fitur) dan y (target), lalu lakukan one-hot encoding pada X.
#
# Petunjuk:
#   y = df["Churn"]
#   X = df.drop(columns=["Churn"])
#   X = pd.get_dummies(X, drop_first=True)  # drop_first untuk hindari multikolinearitas
#   print(X.shape)

# TODO: implementasi


In [ ]:
# === TODO 2.4 — Train-test split ===
# WAJIB pakai stratify=y agar proporsi target seimbang antara train & test.
# Parameter: test_size=0.2, random_state=RANDOM_STATE
#
# Petunjuk:
#   X_train, X_test, y_train, y_test = train_test_split(
#       X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
#   )

# TODO: lakukan train_test_split


# TODO: print shape X_train, X_test, dan proporsi y_train, y_test
#       (untuk verifikasi stratify bekerja)


---

# PART A · Klastering 🧩

**Tujuan:** segmentasi pelanggan menjadi grup-grup berdasarkan karakteristik *(tanpa* label Churn).

> 💡 Klastering BERBEDA dari klasifikasi: tidak ada label saat training. Kita pakai data train **tanpa target** (`X_train`).

### A.1 Pilih fitur & scaling

KMeans sangat sensitif terhadap **skala fitur**. Wajib pakai `StandardScaler`.

In [ ]:
# === TODO A.1 — Scaling fitur untuk klastering ===
# Kita pakai SEMUA kolom X_train (yang sudah di-encode) sebagai input klastering.
#
# Petunjuk:
#   scaler_clust = StandardScaler()
#   X_train_scaled = scaler_clust.fit_transform(X_train)

# TODO: fit & transform


### A.2 Memilih jumlah cluster (Elbow + Silhouette)

Iterasi `K = 2..10`, plot:
- **Inertia** (within-cluster sum of squares) → cari "siku".
- **Silhouette score** → semakin tinggi semakin baik (range -1 sampai 1).

In [ ]:
# === TODO A.2 — Elbow & Silhouette ===
# Loop K dari 2 sampai 10, simpan inertia & silhouette_score.
#
# Petunjuk struktur:
#   k_range = range(2, 11)
#   inertias = []
#   silhouettes = []
#   for k in k_range:
#       km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
#       labels = km.fit_predict(X_train_scaled)
#       inertias.append(km.inertia_)
#       silhouettes.append(silhouette_score(X_train_scaled, labels))

# (a) Implementasi loop di atas
# TODO: tulis kode Anda


# (b) Buat 2 subplot side-by-side: kiri inertia, kanan silhouette (sumbu-x = K)
# Petunjuk: fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# TODO: tulis kode Anda


### TODO A.2 — Pilih K final

> ✏️ **Berdasarkan plot di atas:**
>
> - **K terpilih = ___**
> - **Alasan singkat:** _____

In [ ]:
# === TODO A.3 — Fit KMeans dengan K terpilih ===
# Ganti K_FINAL dengan angka yang Anda pilih di sel sebelumnya.

K_FINAL = ...  # TODO: ganti dengan angka pilihan Anda (mis. 3, 4, atau 5)

# TODO: fit KMeans final dengan K_FINAL, simpan label ke variabel cluster_labels_train
# Petunjuk:
#   km_final = KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init=10)
#   cluster_labels_train = km_final.fit_predict(X_train_scaled)


# TODO: print silhouette score model final


### A.4 Algoritma alternatif

Pilih SALAH SATU pembanding KMeans: **DBSCAN** atau **Agglomerative Clustering**. Bandingkan silhouette score-nya.

In [ ]:
# === TODO A.4 — Algoritma klastering alternatif ===
# Pilih SALAH SATU:
#
# Opsi 1 — Agglomerative (paling mudah):
#   alt = AgglomerativeClustering(n_clusters=K_FINAL)
#   alt_labels = alt.fit_predict(X_train_scaled)
#
# Opsi 2 — DBSCAN (perlu eksperimen eps):
#   alt = DBSCAN(eps=0.5, min_samples=5)
#   alt_labels = alt.fit_predict(X_train_scaled)
#   # ⚠️ DBSCAN bisa menghasilkan label -1 (noise).
#   # Hitung silhouette HANYA pada non-noise:
#   #   mask = alt_labels != -1
#   #   silhouette_score(X_train_scaled[mask], alt_labels[mask])

# (a) Pilih + fit algoritma alternatif
# TODO: tulis kode Anda


# (b) Hitung silhouette score & bandingkan dengan KMeans
# TODO: tulis kode Anda + komentar singkat tentang pemenang


### A.5 Persona tiap cluster

Berikan **interpretasi bisnis**: hitung rata-rata fitur kunci per cluster, lalu beri nama persona.

In [ ]:
# === TODO A.5 — Profil tiap cluster ===
# Tambahkan kolom cluster ke X_train (versi NON-scaled agar mudah diinterpretasi).
#
# Petunjuk:
#   X_train_with_cluster = X_train.copy()
#   X_train_with_cluster["cluster"] = cluster_labels_train
#   profile = X_train_with_cluster.groupby("cluster").mean()
#   display(profile)
#
# Tip: tampilkan hanya 5–7 fitur paling diskriminatif agar mudah dibaca.
# Contoh: tenure, MonthlyCharges, TotalCharges, Contract_Two year, InternetService_Fiber optic

# TODO: bangun & tampilkan tabel profile


# TODO (opsional): heatmap untuk profile dengan sns.heatmap


### TODO A.5 — Beri nama persona

> ✏️ Berdasarkan tabel profil di atas, beri **nama deskriptif** untuk tiap cluster (sesuaikan jumlah baris dengan K_FINAL):
>
> - **Cluster 0:** "_______" — karakteristik utama: _____
> - **Cluster 1:** "_______" — karakteristik utama: _____
> - **Cluster 2:** "_______" — karakteristik utama: _____
> - _(tambahkan jika K > 3)_

### A.6 Visualisasi PCA 2D

Reduksi dimensi ke 2D untuk memvisualisasikan hasil klastering.

In [ ]:
# === TODO A.6 — Visualisasi cluster di ruang PCA 2D ===
# Petunjuk:
#   pca = PCA(n_components=2, random_state=RANDOM_STATE)
#   X_train_pca = pca.fit_transform(X_train_scaled)
#
#   plt.figure(figsize=(8, 6))
#   plt.scatter(X_train_pca[:, 0], X_train_pca[:, 1],
#               c=cluster_labels_train, cmap="tab10", alpha=0.6, s=15)
#   plt.xlabel("PC1"); plt.ylabel("PC2")
#   plt.title(f"KMeans Clusters (K={K_FINAL}) di Ruang PCA 2D")
#   plt.colorbar(label="Cluster")
#   plt.show()

# TODO: implementasi


# TODO: print pca.explained_variance_ratio_
#       → berapa % total variansi yang dijelaskan 2 PC pertama?


---

# PART B · Klasifikasi 🎯

**Tujuan:** memprediksi apakah pelanggan akan churn berdasarkan fitur-fiturnya.

> 💡 Berbeda dari Part A — di sini kita PAKAI label `y_train` untuk *supervised learning*.

### B.1 Baseline: Logistic Regression

Model linear, interpretable. Ekspektasinya jadi *floor* performa.

In [ ]:
# === TODO B.1 — Logistic Regression ===
# 🚨 Penting: scaling FIT hanya di X_train, TRANSFORM (bukan fit_transform) di X_test.
#    Kalau Anda fit_transform di X_test → DATA LEAKAGE → nilai dipotong.
#
# Petunjuk:
#   scaler_clf = StandardScaler()
#   X_train_clf = scaler_clf.fit_transform(X_train)
#   X_test_clf  = scaler_clf.transform(X_test)
#
#   logreg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
#   logreg.fit(X_train_clf, y_train)
#   y_pred_lr  = logreg.predict(X_test_clf)
#   y_proba_lr = logreg.predict_proba(X_test_clf)[:, 1]

# TODO: implementasi sesuai petunjuk


### B.2 Model non-linear: Random Forest

Random Forest **tidak butuh scaling** karena berbasis tree. Bandingkan dengan LogReg.

In [ ]:
# === TODO B.2 — Random Forest ===
# Petunjuk:
#   rf = RandomForestClassifier(
#       n_estimators=200,
#       random_state=RANDOM_STATE,
#       n_jobs=-1,
#   )
#   rf.fit(X_train, y_train)              # tidak perlu scaling
#   y_pred_rf  = rf.predict(X_test)
#   y_proba_rf = rf.predict_proba(X_test)[:, 1]

# TODO: implementasi


### B.3 Evaluasi & Perbandingan

Karena kelas tidak seimbang, **bukan hanya accuracy**. Wajib tampilkan:
- Confusion matrix
- Precision, Recall, F1 (dari `classification_report`)
- ROC-AUC

In [ ]:
# === TODO B.3 — Evaluasi kedua model ===
# Buat fungsi helper agar tidak duplicate code.
#
# Petunjuk:
#   def evaluate(name, y_true, y_pred, y_proba):
#       print(f"=== {name} ===")
#       print(classification_report(y_true, y_pred))
#       print(f"ROC-AUC: {roc_auc_score(y_true, y_proba):.4f}")
#       ConfusionMatrixDisplay.from_predictions(y_true, y_pred, cmap="Blues")
#       plt.title(name)
#       plt.show()

# (a) Definisikan fungsi evaluate
# TODO: tulis kode Anda


# (b) Panggil evaluate untuk LogReg & RandomForest
# TODO: tulis kode Anda


# (c) BONUS — plot ROC curve kedua model di satu figure
# Petunjuk:
#   fig, ax = plt.subplots(figsize=(7, 6))
#   RocCurveDisplay.from_predictions(y_test, y_proba_lr, name="LogReg", ax=ax)
#   RocCurveDisplay.from_predictions(y_test, y_proba_rf, name="RandomForest", ax=ax)
#   ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
#   plt.show()


### TODO B.3 — Refleksi pemilihan model

> ✏️ Jawab semua pertanyaan berikut:
>
> 1. **Model mana yang lebih baik?** Sebutkan metrik yang Anda jadikan acuan dan **alasannya** (bukan sekadar angka tertinggi).
> 2. **Trade-off apa** yang Anda lihat antara LogReg vs RF (interpretability, training time, performa, dll.)?
> 3. **Untuk kasus churn**, metrik mana yang paling penting menurut tim Anda — **Recall**? **Precision**? **F1**? Mengapa? (Petunjuk: pikirkan biaya kesalahan: false positive vs false negative.)

---

# PART C · Integrasi 🔗

**Ide:** label cluster dari Part A bisa dijadikan **fitur tambahan** di Part B. Apakah ini meningkatkan performa? Apa risikonya?

> 🚨 **DANGER ZONE — Anti data leakage!**  
> Klastering HARUS di-`fit` hanya pada **`X_train`**. Untuk `X_test`, kita PAKAI `.predict()` pada model klastering yang sudah di-fit di train.
>
> **JANGAN** fit ulang KMeans pada data gabungan train+test — itu data leakage.

In [ ]:
# === TODO C.1 — Refit KMeans di X_train, predict di X_test (anti-leakage) ===
# Petunjuk:
#   # Scale X_test pakai scaler_clust yang SUDAH di-fit di train (jangan fit ulang!)
#   X_test_scaled = scaler_clust.transform(X_test)
#
#   km_for_feat = KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init=10)
#   train_clusters = km_for_feat.fit_predict(X_train_scaled)   # FIT di train
#   test_clusters  = km_for_feat.predict(X_test_scaled)        # hanya PREDICT di test

# TODO: implementasi (pastikan fit hanya di X_train_scaled!)


In [ ]:
# === TODO C.2 — Tambahkan cluster ID sebagai feature & retrain RF ===
# Petunjuk:
#   X_train_plus = X_train.copy()
#   X_train_plus["cluster"] = train_clusters
#   X_test_plus  = X_test.copy()
#   X_test_plus["cluster"]  = test_clusters
#
#   rf_plus = RandomForestClassifier(
#       n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
#   )
#   rf_plus.fit(X_train_plus, y_train)
#   y_pred_rf_plus  = rf_plus.predict(X_test_plus)
#   y_proba_rf_plus = rf_plus.predict_proba(X_test_plus)[:, 1]

# TODO: implementasi


In [ ]:
# === TODO C.3 — Bandingkan dengan Random Forest tanpa cluster feature ===
# Tampilkan side-by-side metrik (F1, ROC-AUC) antara RF (Part B) dan RF+cluster (Part C).
# Petunjuk:
#   - Pakai fungsi evaluate() dari B.3 untuk dua model, ATAU
#   - Bangun DataFrame perbandingan:
#       from sklearn.metrics import f1_score
#       comparison = pd.DataFrame({
#           "Model": ["RF (B)", "RF + cluster (C)"],
#           "F1":      [f1_score(y_test, y_pred_rf), f1_score(y_test, y_pred_rf_plus)],
#           "ROC-AUC": [roc_auc_score(y_test, y_proba_rf), roc_auc_score(y_test, y_proba_rf_plus)],
#       })
#       display(comparison)

# TODO: tampilkan perbandingan metrik


# TODO (opsional): lihat feature_importances_ rf_plus → di urutan ke berapa fitur 'cluster'?


### TODO C.4 — Refleksi integrasi

> ✏️ Jawab semua pertanyaan berikut:
>
> 1. **Apakah ada peningkatan?** Berikan angka konkret (mis. "F1 naik dari 0.58 ke 0.61").
> 2. **Jika TIDAK ada peningkatan / malah turun**, mengapa? (Petunjuk: lihat `feature_importances_`. Mungkin cluster ID redundan dengan fitur asli yang sudah ada.)
> 3. **Risiko data leakage apa** yang masih mungkin terjadi meskipun kita sudah hati-hati di sel C.1? (Petunjuk: pikirkan preprocessing sebelum split, atau encoding kategorik yang melibatkan target.)
> 4. **Kapan secara umum** menggabungkan klastering ke klasifikasi memberi manfaat di praktik nyata?

---

## 🏁 Kesimpulan & Refleksi Akhir Kelompok

> ✏️ **Jawab MINIMAL 3 dari 5 pertanyaan berikut** (juga salin ke `reports/laporan.md`):
>
> 1. Algoritma klastering mana yang paling cocok untuk dataset ini, dan apa indikatornya?
> 2. Algoritma klasifikasi mana yang Anda pilih untuk diproduksi, dan apa pertimbangan bisnisnya?
> 3. Apa hal terpenting yang kelompok Anda pelajari dari **perbedaan paradigma supervised vs unsupervised**?
> 4. Bagian mana dari pipeline yang menurut Anda paling rawan error / leakage di praktik nyata?
> 5. Jika diberi waktu 1 minggu lagi, eksperimen / perbaikan apa yang akan Anda coba?

---

🎉 **Selamat! Anda sudah menyelesaikan tugas.**

Checklist sebelum submit:
- ✅ Semua sel `# TODO` sudah diisi & ter-eksekusi tanpa error.
- ✅ `reports/laporan.md` lengkap (semua bagian terisi).
- ✅ Notebook sudah di-`Save a copy in GitHub` ke repo Classroom tim.
- ✅ Commit terakhir di-push **sebelum deadline**.